In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Save pre-labeled data
with open(f'./labeled-issues/prelabeled_issues_1.json') as f:
    labeled_issues_1 = json.load(f)

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# Data Preparation
labeled_issues_list = list(labeled_issues_1.values()) 
train_issues, test_issues = train_test_split(labeled_issues_list, test_size=0.2)
twenty_true_labels = [issue["ground_truth_label"] for issue in test_issues]
eighty_true_labels = [issue["ground_truth_label"] for issue in train_issues]

In [67]:
def get_llm_prediction(issues):
    

    prompt = """
    Classify this GitHub issue into ONE of these WCAG violation categories:

    1. color-contrast - Text/background contrast ratio below 4.5:1 (e.g., "button text too light")
    2. missing-alt-text - Images/icons without alt text (e.g., "<img src='logo.png'>")
    3. keyboard-navigation - Non-keyboard-operable elements (e.g., "can't tab to dropdown")
    4. aria-misuse - Incorrect ARIA roles/states (e.g., "role='button' missing aria-pressed")
    5. focus-management - Missing focus indicators or broken focus order (e.g., "focus invisible on buttons")
    6. semantic-html - Misused HTML tags (e.g., "<div onclick> instead of <button>")
    7. mobile-responsive - Touch targets <44px or viewport issues (e.g., "checkbox too small on mobile")
    8. motion-risk - Auto-playing motion without controls (e.g., "carousel auto-advances too fast")
    9. language-errors - Missing lang attributes (e.g., "<html> missing lang='en'")
    10. skip-link - Missing "Skip to Content" links (e.g., "no way to skip navigation")
    11. pdf-accessibility - Untagged PDFs (e.g., "PDF has no headings")
    12. captions-audio - Missing captions/transcripts (e.g., "video has no subtitles")
    13. other - General accessibility issues not covered above
    14. none - if the issue is not related to accessibility, or body is empty
    15. pointer-issues - hovering/scrolling/clicking issues (e.g., "hover menu not working")

    Respond ONLY with the label name. Example: "missing-alt-text"

    ISSUE TITLE: {title}
    ISSUE BODY: {body}
    """
    
    result = []
    count = 1
    for issue in issues: 
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt.format(title=issue['title'], body=issue['body']),
        )
        prediction = response.output[0].content[0].text
        result.append(prediction)
        print(f"Added issue label: {prediction}, count: {count}")
        count += 1
        
    print("done with prediction")
    return result


In [61]:
twenty_llm_predictions = get_llm_prediction(test_issues)

Added issue label: none, count: 1
Added issue label: none, count: 2
Added issue label: focus-management, count: 3
Added issue label: none, count: 4
Added issue label: aria-misuse, count: 5
Added issue label: other, count: 6
Added issue label: color-contrast, count: 7
Added issue label: none, count: 8
Added issue label: none, count: 9
Added issue label: aria-misuse, count: 10
Added issue label: missing-alt-text, count: 11
Added issue label: none, count: 12
Added issue label: other, count: 13
Added issue label: semantic-html, count: 14
Added issue label: none, count: 15
Added issue label: focus-management, count: 16
Added issue label: none, count: 17
Added issue label: none, count: 18
Added issue label: keyboard-navigation, count: 19
Added issue label: none, count: 20
done with prediction


In [62]:
score = accuracy_score(twenty_true_labels, twenty_llm_predictions)

score

0.9

In [63]:
print(classification_report(twenty_true_labels, twenty_llm_predictions))

                     precision    recall  f1-score   support

        aria-misuse       1.00      0.67      0.80         3
     color-contrast       1.00      1.00      1.00         1
   focus-management       1.00      0.67      0.80         3
keyboard-navigation       1.00      1.00      1.00         1
   missing-alt-text       1.00      1.00      1.00         1
               none       0.90      1.00      0.95         9
              other       0.50      1.00      0.67         1
      semantic-html       1.00      1.00      1.00         1

           accuracy                           0.90        20
          macro avg       0.93      0.92      0.90        20
       weighted avg       0.93      0.90      0.90        20



In [68]:
eighty_llm_predictions = get_llm_prediction(train_issues)

Added issue label: color-contrast, count: 1
Added issue label: color-contrast, count: 2
Added issue label: none, count: 3
Added issue label: none, count: 4
Added issue label: none, count: 5
Added issue label: pointer-issues, count: 6
Added issue label: none, count: 7
Added issue label: color-contrast, count: 8
Added issue label: keyboard-navigation, count: 9
Added issue label: none, count: 10
Added issue label: mobile-responsive, count: 11
Added issue label: keyboard-navigation, count: 12
Added issue label: color-contrast, count: 13
Added issue label: none, count: 14
Added issue label: none, count: 15
Added issue label: other, count: 16
Added issue label: color-contrast, count: 17
Added issue label: captions-audio, count: 18
Added issue label: focus-management, count: 19
Added issue label: none, count: 20
Added issue label: semantic-html, count: 21
Added issue label: none, count: 22
Added issue label: color-contrast, count: 23
Added issue label: keyboard-navigation, count: 24
Added iss

In [69]:
score = accuracy_score(eighty_true_labels, eighty_llm_predictions)

score

0.8

In [70]:
print(classification_report(eighty_true_labels, eighty_llm_predictions))


                     precision    recall  f1-score   support

        aria-misuse       0.75      0.75      0.75         4
     captions-audio       1.00      1.00      1.00         1
     color-contrast       0.91      0.91      0.91        11
   focus-management       0.75      0.75      0.75         4
keyboard-navigation       1.00      1.00      1.00         9
    language-errors       1.00      1.00      1.00         1
   missing-alt-text       1.00      1.00      1.00         1
  mobile-responsive       0.50      1.00      0.67         1
               none       0.78      1.00      0.88        28
              other       0.67      0.18      0.29        11
  pdf-accessibility       1.00      1.00      1.00         1
     pointer-issues       1.00      0.50      0.67         4
      semantic-html       0.50      0.50      0.50         4
          skip-link       0.00      0.00      0.00         0

           accuracy                           0.80        80
          macro avg   

C:\Users\Myself\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Myself\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Myself\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1531: U